# Transaction Foundation Model on Ray — Part 3: Tokenize with NVIDIA's tokenizer

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: ~5 min (full: longer — encodes the whole train split)

---

Previously in Part 2, we built the train/validation/test splits of the raw transactions. In this notebook we tokenize the training split — we convert each transaction into token ids, the only input a transformer can train on. Part 4 then pretrains the foundation model on the result.

Tokenizing transactions works differently than tokenizing text. An LLM chops free-form text into word pieces. A transaction already comes in fields — an amount, a merchant, an hour — so each field maps directly to a token: 6 AM is `HOUR_06` (one of 24 hour tokens), \$57.20 is the \$50–\$100 amount token (one of 7), a merchant is one of 2,000 merchant tokens. Twelve tokens cover a whole transaction. The mapping is mechanical; designing it is the hard part — how finely to bucket amounts, how many merchant tokens, which fields get tokens at all. Those choices form the fixed vocabulary, and they decide what the model can ever learn.

The language analogy holds where it matters: a transaction is a sentence, a card's history is a document. We lay each card's transactions end to end as one token stream, cut it into fixed-length windows, and those windows are the corpus Part 4 trains on.

The tokenizer is NVIDIA's, unchanged. We run it distributed across CPU workers with Ray Data.

In [1]:
import sys, os, json

DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import numpy as np
import pandas as pd
import logging

import ray
ray.init(ignore_reinit_error=True, runtime_env={"working_dir": DEMO_ROOT},
         logging_level=logging.ERROR)

from src.paths import artifact_paths, get_demo_base_dir
from src.scale_config import load_scale

# Configure the 'scale' of the data we work with
SCALE = "mini"             # mini = 200 card-holders on CPU, minutes; full = every card
cfg = load_scale(SCALE)
paths = artifact_paths(get_demo_base_dir(), SCALE)

## The tokenizer's design choices

These are the choices NVIDIA's tokenizer makes. The Part 2 distributions explain each one.

- **Amount** — bucketed at fixed dollar thresholds (\$0/10/50/100/500/1K/5K), one token per bucket. Part 2 showed amounts spanning orders of magnitude; the thresholds sit roughly one per factor of ten so the buckets match the spread.
- **Merchant** — ~91K distinct names, far too many for one token each, so names are hashed into 2,000 buckets. Hashing keeps the vocabulary bounded; collisions (two merchants sharing a token) are the accepted cost.
- **MCC** — the merchant category code (`5411` is grocery stores). It becomes two tokens: the exact code (`MCC_5411`) and a broader industry token derived from it (`CAT_RETAIL`). The broad token lets the model generalize across related merchants even when an exact code is rare.
- **Time** — hour, day-of-week, and month tokens on every transaction. Part 2 showed the gaps are bursty, and position in the sequence says nothing about elapsed time, so time gets its own tokens. (NVIDIA also ships an optional gap-since-last-transaction tokenizer; the blueprint leaves it off, and so do we.)
- **Everything else** — the remaining fields have only a handful of possible values, so each value gets its own token: which of the customer's cards it is, the channel (chip, swipe, or online), the merchant's ZIP region and state, and the customer id.

Each transaction is now twelve tokens. We add a `<sep>` token between transactions, so a transaction takes about 13 positions in its card's stream. A 4096-token window therefore holds about 315 transactions of context; at `mini` the windows are 256 tokens, about 19 transactions.

Add up every possible token across the fields and you get the whole vocabulary: 6,251 ids, fixed before the model ever sees data. Next we build these windows for every card in the training split.

## Build the training windows

We build the training windows with one Ray Data job. Each card can be tokenized on its own — its tokens come only from its own transactions — so the job groups the training split by card and tokenizes the cards in parallel across the CPU workers. The grouping shuffle is the heavy distributed work; the per-card tokenizing is a plain pandas function.

That pandas function is NVIDIA's tokenizer, translated from cuDF (their GPU version of pandas) to plain pandas in `src/nvtokenize_cpu.py` — tokenization is string formatting and table lookups, and cheap CPU cores do that fine. The translation is verified byte-identical to NVIDIA's original (the checks live in `scripts/`), so both models train on exactly the same tokens and Part 6's comparison against their published numbers stays fair.

In [2]:
from src.nvsplit import train_parquet_files
from src.nvcorpus import tokenize_card_group, fresh_seqs_dir, assemble_sequences, wait_for_files
import shutil

tk = cfg["tokenize"]
seq_len = tk["seq_len"]                  # tokens per window (4096 at full — NVIDIA's setting)
chunk = max(1, seq_len // 13)            # ~13 positions per transaction -> transactions per window
max_seq = tk.get("max_pretrain_windows") # cap the number of windows (mini); None = keep all

if not os.path.exists(os.path.join(paths["nvcorpus"], "ids.npy")):
    seqs_dir = fresh_seqs_dir(paths["nvcorpus"])   # temp output dir for the per-card shards

    # Group the training split by card, then run tokenize_card_group on each card:
    # order its rows by time, tokenize them, pack them into seq_len windows.
    # One pandas call per card, in parallel across the CPU workers.
    ray.data.read_parquet(train_parquet_files(paths["nvsplit"])) \
        .groupby(["User", "Card"]) \
        .map_groups(lambda g: tokenize_card_group(g, seq_len, chunk),
                    batch_format="pandas") \
        .write_parquet(seqs_dir)

    # assemble_sequences is a Ray task: it packs the shards into ids.npy / attn.npy /
    # vocab.json in a fixed (user, card, chunk) order, so every build is identical.
    meta = ray.get(assemble_sequences.remote(seqs_dir, paths["nvcorpus"], seq_len, max_seq))
    wait_for_files([os.path.join(paths["nvcorpus"], f)     # shared storage: a worker's
                    for f in ("ids.npy", "attn.npy")])     # write can lag the driver's view
    shutil.rmtree(seqs_dir)                  # the shards were only inputs to the assembly
    print(json.dumps(meta, indent=2))
else:
    print("corpus already built at", paths["nvcorpus"])

corpus already built at /mnt/cluster_storage/transaction-fm/nvcorpus/mini/


## Check the windows

The windows are on shared storage now. We check its size and how much of it is real tokens versus padding, then decode the first window back into readable tokens with the same tokenizer. (Importing the tokenizer pulls in cuDF, which prints a warning that it finds no GPU. That is expected — everything here runs on CPU.)

In [3]:
from src.nvidia_tokenizer import FinancialTabularTokenizer

# Load the corpus. mmap reads the arrays without pulling them fully into memory.
ids = np.load(os.path.join(paths["nvcorpus"], "ids.npy"), mmap_mode="r")
attn = np.load(os.path.join(paths["nvcorpus"], "attn.npy"), mmap_mode="r")
vocab_info = json.load(open(os.path.join(paths["nvcorpus"], "vocab.json")))

print(f"{ids.shape[0]:,} windows × {ids.shape[1]} tokens  "
      f"(vocab {vocab_info['vocab_size']:,}, pad id {vocab_info['pad']})")
print(f"real tokens (not padding): {float(np.asarray(attn).mean()):.1%}\n")

# FinancialTabularTokenizer is NVIDIA's tokenizer: it converts transactions to
# tokens and back. We use the "back" direction here — decode the first window
# into readable tokens. The settings match the corpus build.
tok = FinancialTabularTokenizer(merchant_hash_size=2000, category_hierarchy=True,
                                temporal_encoding=True)
window = np.asarray(ids[0])
tokens = [int(t) for t in window if int(t) != vocab_info["pad"]]   # drop the padding
decoded = tok.decode(tokens).split()
print(f"window 0 — {len(tokens)} real tokens; the first two transactions:")
print("  " + " ".join(decoded[:27]))

8 windows × 256 tokens  (vocab 6,251, pad id 0)
real tokens (not padding): 96.9%

window 0 — 248 real tokens; the first two transactions:
  <bos> AMT_3 MERCH_667 CAT_RETAIL MCC_5300 HOUR_06 DOW_6 MONTH_09 CARD_0 CHIP_SWIPE ZIP3_917 STATE_CA CUST_0 <sep> AMT_1 MERCH_869 CAT_RETAIL MCC_5411 HOUR_06 DOW_6 MONTH_09 CARD_0 CHIP_SWIPE ZIP3_917 STATE_CA CUST_0 <sep>


The 6,251 ids are organized in blocks: the five special tokens come first, then each field gets its own block. The cell below counts them. The count is fixed, so Part 4 can size the model before it sees any data.

In [4]:
from collections import Counter

vocab = tok.vocab   # same tokenizer instance as above
specials = [t for t in ("<pad>", "<bos>", "<eos>", "<sep>", "<unk>") if t in vocab]
print(f"vocabulary: {tok.vocab_size:,} tokens  ({len(specials)} special + the per-field blocks)")
print(f"special tokens: {specials}\n")

# Count each field's block by its token prefix (AMT_, MERCH_, HOUR_, ...).
per_field = Counter(t.split("_")[0] for t in vocab if t not in specials and "_" in t)
print("tokens per field:")
for name, size in sorted(per_field.items(), key=lambda kv: -kv[1]):
    print(f"  {name:<12} {size:>6,}")

vocabulary: 6,251 tokens  (5 special + the per-field blocks)
special tokens: ['<pad>', '<bos>', '<eos>', '<sep>', '<unk>']

tokens per field:
  CUST          3,000
  MERCH         2,000
  ZIP3          1,000
  MCC             110
  STATE            58
  HOUR             24
  CAT              14
  MONTH            12
  CARD             10
  AMT               7
  DOW               7
  CHIP              4


## Scaling factors

This job has two main steps: grouping the rows by card, then tokenizing each card. They scale in different ways.

Grouping is limited by how fast you can move data around. All of a card's rows have to sit on the same machine before we can tokenize that card. On one machine, that means the whole table sits in RAM — the same wall as Part 2, and a groupby hits it sooner, somewhere past a few hundred million rows. On a cluster, it means sending nearly every row over the network to the machine that owns its card — data engineers call this a shuffle, and at production scale it is terabytes of network traffic. Ray Data streams the shuffle in blocks: each worker receives only the groups it is working on right now, and if memory runs short, Ray spills blocks to disk rather than crashing.

Tokenizing scales directly with how many cpu cores you give it (it's 'embarrassingly parallel'). Cards are independent, so every task can work in isolation: a 32-core machine tokenizes 32 cards at a time, a pool of workers tokenizes as many as it has cores, and the pool size is one config value. Double the workers and the wall-clock halves, until the shuffle becomes the bottleneck. This is also why production scale is not a problem here — tens of millions of cards is just more independent work.

The output stays manageable: about 1 GB at full scale, growing linearly with the transactions. `seq_len` and `max_pretrain_windows` trade context length and output size against training cost.

The code never changed across any of this: the same five-line job ran at `mini` and `full`, with only the config and the worker pool differing.

## Takeaways

We built the pretraining data. Every card's transactions are tokenized, packed into windows, and written to shared storage as three files — `ids.npy`, `attn.npy`, and `vocab.json`. Part 4 trains on these files directly, and opens by loading them — we'll look at what's inside there.

We used Ray Data's `groupby → map_groups` for the build: the shuffle brought each card's rows together, and the tokenizing ran as plain pandas across the CPU workers. One Ray task assembled the output files. The tokenizer is NVIDIA's, translated to pandas and verified byte-identical, so our training data matches theirs exactly.

---

## Next

**Part 4 — Pretrain**: train the foundation model on these windows, with Ray Train running the training loop — one CPU worker at `mini`, eight GPUs at `full`.